# Phase II · A2C Prompt Selector — Training & Analysis

**Algorithm:** Advantage Actor-Critic (A2C)  

---

## How A2C Differs from DQN

| Aspect | DQN | A2C |
|--------|-----|-----|
| Policy type | Implicit (argmax Q) | Explicit softmax π(a|s) |
| Critic | Q-values per action | Single scalar V(s) |
| Updates | Off-policy (replay buffer) | On-policy (n-step rollouts) |
| Exploration | ε-greedy decay | Entropy bonus H(π) |
| Sample efficiency | Higher (replay) | Lower (on-policy) |
| Stability | Can oscillate with large updates | More stable once converged |

**Advantage function:**
$$A(s_t, a_t) = R_t - V(s_t)$$

where $R_t = r_t + \gamma r_{t+1} + \cdots + \gamma^{T-t} V(s_T)$ is the n-step return.

**Policy gradient loss:**
$$L_{\text{policy}} = -\mathbb{E}\left[\log \pi(a_t|s_t) \cdot A(s_t, a_t)\right]$$

**Total loss:**
$$L = L_{\text{policy}} + c_v \cdot L_{\text{value}} - c_e \cdot H(\pi)$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from mdp_phase2.environment      import MDPCustomerServiceEnv
from mdp_phase2.agents.a2c_agent import A2CAgent
from mdp_phase2.reward           import RewardShaper
from mdp_phase2.train            import train_a2c, evaluate, run_demo_conversation, TrainConfig
from mdp_phase2.strategy_prompts import STRATEGY_NAMES, STRATEGY_LABELS, STRATEGY_COLORS

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = list(STRATEGY_COLORS.values())

print('Imports OK ✓')

## 1 · A2C Architecture

In [ ]:
def make_a2c():
    return A2CAgent(
        state_dim     = MDPCustomerServiceEnv.STATE_DIM,
        num_actions   = MDPCustomerServiceEnv.NUM_ACTIONS,
        gamma         = 0.90,
        lr            = 5e-4,
        n_steps       = 8,      # update every 8 steps (within episode)
        c_value       = 0.50,   # value loss weight
        c_entropy     = 0.02,   # entropy bonus (encourages exploration)
        gradient_clip = 0.50,
        hidden        = (128, 64),
    )

agent_tw = make_a2c()

try:
    agent_tw._model.summary()
except:
    print('A2C Agent (shared Actor-Critic trunk)')
    print('  Architecture : 15 → 128 → 64 → (Actor: 5) + (Critic: 1)')

print(f'\nc_entropy (exploration bonus): {agent_tw.c_entropy}')
print(f'n_steps (rollout length)     : {agent_tw.n_steps}')

## 2 · Training — All Three Datasets

In [ ]:
cfg = TrainConfig(
    n_episodes    = 3000,
    eval_every    = 100,
    eval_episodes = 50,
    print_every   = 300,
    seed          = 42,
)

agent_tw = make_a2c()
result_tw = train_a2c(agent_tw, 'twitter', cfg)

agent_rd = make_a2c()
result_rd = train_a2c(agent_rd, 'reddit', cfg)

agent_oa = make_a2c()
result_oa = train_a2c(agent_oa, 'openassistant', cfg)

## 3 · Learning Curves & Strategy Evolution

In [ ]:
results    = {'Twitter': result_tw, 'Reddit': result_rd, 'OpenAssistant': result_oa}
agents_map = {'Twitter': agent_tw,  'Reddit': agent_rd,  'OpenAssistant': agent_oa}
ds_colors  = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (ds_name, res) in enumerate(results.items()):
    # row 0: smoothed reward
    ax = axes[0, col]
    smooth = res.smooth_rewards(50)
    offset = len(res.episode_rewards) - len(smooth)
    ax.plot(res.episode_rewards, alpha=0.2, color=ds_colors[col], linewidth=0.5)
    ax.plot(range(offset, offset+len(smooth)), smooth, color=ds_colors[col], linewidth=2)
    ax.set_title(f'Reward — {ds_name}', fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')

    # row 1: strategy usage evolution (stacked area, sampled at eval points)
    ax = axes[1, col]
    eval_eps = [ev.episode for ev in res.eval_snapshots]
    usage_over_time = np.array([[ev.strategy_dist.get(n, 0)*100
                                  for n in STRATEGY_NAMES]
                                 for ev in res.eval_snapshots])
    ax.stackplot(eval_eps,
                 [usage_over_time[:, i] for i in range(len(STRATEGY_NAMES))],
                 labels=STRATEGY_LABELS, colors=COLORS, alpha=0.8)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Strategy usage (%)')
    ax.set_title(f'Strategy Mix — {ds_name}', fontweight='bold')
    if col == 2:
        ax.legend(loc='upper right', fontsize=7)

plt.suptitle('A2C Training Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('a2c_training_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Policy Entropy During Training

Entropy measures how spread out the policy is.  
- **High entropy (early)**: agent explores many strategies equally  
- **Low entropy (late)**: agent has converged to a preferred strategy for each state

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

max_entropy = np.log(MDPCustomerServiceEnv.NUM_ACTIONS)  # log(5) ≈ 1.609

for ax, (ds_name, res), color in zip(axes, results.items(), ds_colors):
    # approximate entropy from eval strategy distributions
    entropies = []
    eval_eps  = []
    for ev in res.eval_snapshots:
        dist = np.array([ev.strategy_dist.get(n, 1e-8) for n in STRATEGY_NAMES])
        dist = dist / dist.sum()
        H    = -np.sum(dist * np.log(dist + 1e-8))
        entropies.append(H / max_entropy * 100)  # normalised to 0–100%
        eval_eps.append(ev.episode)

    ax.plot(eval_eps, entropies, color=color, linewidth=2, marker='o', ms=4)
    ax.axhline(100, linestyle='--', color='gray', alpha=0.5, label='Max entropy (uniform)')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Normalised Entropy (%)')
    ax.set_title(f'Policy Entropy — {ds_name}', fontweight='bold')
    ax.set_ylim(0, 115)
    ax.legend(fontsize=8)

plt.suptitle('A2C Policy Entropy Over Training  (100% = uniform random)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('a2c_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · Demo Conversation — A2C Agent

In [ ]:
demo = run_demo_conversation(agent_tw, dataset='twitter', verbose=True)

## 6 · Summary

In [ ]:
print('=' * 60)
print('  A2C PHASE II — TRAINING SUMMARY')
print('=' * 60)
for ds_name, res in results.items():
    ev = res.eval_snapshots[-1]
    print(f'\n  {ds_name}')
    print(f'    Resolution rate : {ev.success_rate*100:.1f}%')
    print(f'    Escalation rate : {ev.escalation_rate*100:.1f}%')
    print(f'    Avg reward      : {ev.avg_reward:+.3f}')
    print(f'    Avg turns       : {ev.avg_turns:.1f}')
    print(f'    Training time   : {res.train_time_s:.1f}s')
print('\n  See comparison_phase2.ipynb for full DQN vs A2C vs PPO analysis')
print('=' * 60)